# Kubernetes Replay Analysis

Run the HetPolicy and CarbonScaler aggregates, review the KPIs, and regenerate the Pareto artefacts.

In [ ]:
from pathlib import Path
import pandas as pd
import plotly.express as px

from analysis.scripts.aggregate_k8s import run, DEFAULT_E_REF, DEFAULT_C_REF

repo_root = Path('..', '..').resolve()
het_path = repo_root / 'kubenergysched' / 'results_k8s' / 'hetpolicy' / 'decisions.jsonl'
carb_path = repo_root / 'kubenergysched' / 'results_k8s' / 'carbonscaler' / 'decisions.jsonl'
output_dir = repo_root / 'analysis' / 'k8s_results'
figures_dir = repo_root / 'analysis' / 'figures' / 'k8s'
workloads_csv = repo_root / 'kubenergysched' / 'config' / 'workloads.csv'

bundle = run(
    het_path=het_path,
    carb_path=carb_path,
    output_dir=output_dir,
    workloads_path=workloads_csv,
    e_ref=DEFAULT_E_REF,
    c_ref=DEFAULT_C_REF,
    figures_dir=figures_dir,
)
bundle

### Policy summary

In [ ]:
summary_df = pd.DataFrame(bundle['summary']).set_index('policy')
summary_df.round(6)

### Pareto frontier

In [ ]:
pareto_df = pd.DataFrame(bundle['pareto'])
pareto_df.round(6)

### Per-job measurements

In [ ]:
per_job_df = pd.read_csv(output_dir / 'per_job_combined.csv')
per_job_df.head()

### Carbon vs makespan

In [ ]:
fig = px.scatter(
    summary_df.reset_index(),
    x='total_carbon_kg',
    y='makespan_s',
    color='policy',
    size='jobs',
    text='policy',
    hover_data={
        'avg_wait_s': ':,.2f',
        'avg_runtime_s': ':,.2f',
        'throughput_jobs_per_hour': ':,.2f',
    },
    labels={
        'total_carbon_kg': 'Total carbon footprint (kg)',
        'makespan_s': 'Makespan (s)'
    },
    title='Kubernetes Pareto – Carbon vs Makespan',
)
fig.update_traces(textposition='top right')
fig.update_layout(legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1.0))
fig.show()
png_path = figures_dir / 'k8s_pareto_carbon_makespan.png'
html_path = figures_dir / 'k8s_pareto_carbon_makespan.html'
try:
    fig.write_image(str(png_path), scale=2, width=960, height=640)
except ValueError as exc:
    print(f'PNG export skipped: {exc}. Install `kaleido` to enable static export.')
fig.write_html(str(html_path), include_plotlyjs='cdn')
png_path